# BidVision Pilot — Temiz Kurulum

Bu notebook, RAM++ ile etiket çıkarımı ve Grounded-SAM-2 (Grounding DINO + SAM2) ile konum/maske çıkarımını uçtan uca çalıştırır.

**Kullanmadan önce:**
1. Sağ panelden **Accelerator → GPU** seç.
2. Sağ panelden **Add Input** ile kendi depo fotoğraflarını içeren bir dataset ekle, aşağıdaki `IMG_PATH` değerlerini kendi dosya yoluna göre güncelle.
3. Hücreleri sırayla, yukarıdan aşağı çalıştır (Run All ile de çalışabilir).

## Bölüm 1 — RAM++ Kurulumu

In [1]:
!git clone https://github.com/xinyu1205/recognize-anything.git
%cd recognize-anything

Cloning into 'recognize-anything'...
fatal: unable to access 'https://github.com/xinyu1205/recognize-anything.git/': Could not resolve host: github.com
[Errno 2] No such file or directory: 'recognize-anything'
/kaggle/working


In [2]:
!pip install -e . -q
!pip install "transformers==4.46.3" -q
!pip install fairscale -q
!pip install git+https://github.com/openai/CLIP.git -q

ERROR: file:///kaggle/working does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.
ERROR: Could not find a version that satisfies the requirement transformers==4.46.3 (from versions: none)
ERROR: No matching distribution found for transformers==4.46.3
ERROR: Could not find a version that satisfies the requirement fairscale (from versions: none)
ERROR: No matching distribution found for fairscale
  error: subprocess-exited-with-error
  
  × git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-oouav4hn did not run successfully.
  │ exit code: 128
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: subprocess-exited-with-error

× git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-oouav4hn did not run successfully.
│ exit code: 128
╰─> See above for output.

note: This error originates from a subprocess, and

Kaggle'ın hazır getirdiği `transformers` sürümü çok yeni olduğu için, RAM'in `bert.py` dosyasındaki bazı importlar eski konumdan (`modeling_utils`) yeni konuma (`pytorch_utils`) taşınmış. Aşağıdaki hücre bunu otomatik düzeltir.

In [3]:
path = "ram/models/bert.py"
with open(path, "r") as f:
    content = f.read()

old = """from transformers.modeling_utils import (
    PreTrainedModel,
    apply_chunking_to_forward,
    find_pruneable_heads_and_indices,
    prune_linear_layer,
)"""

new = """from transformers.modeling_utils import PreTrainedModel
from transformers.pytorch_utils import (
    apply_chunking_to_forward,
    find_pruneable_heads_and_indices,
    prune_linear_layer,
)"""

if old in content:
    content = content.replace(old, new)
    with open(path, "w") as f:
        f.write(content)
    print("bert.py düzeltildi")
else:
    print("Eşleşme bulunamadı — dosya değişmiş olabilir, elle kontrol et")

FileNotFoundError: [Errno 2] No such file or directory: 'ram/models/bert.py'

In [ ]:
!mkdir -p pretrained
!wget -q https://huggingface.co/xinyu1205/recognize-anything-plus-model/resolve/main/ram_plus_swin_large_14m.pth -P pretrained/
print("RAM++ ağırlığı indirildi")

Kendi kısıtlı sınıf listemizle çalışan basit bir çıkarım script'i yazıyoruz. `my_categories` listesini kendi taksonominize göre güncelleyebilirsiniz.

In [ ]:
%%writefile inference_custom.py
import numpy as np
import torch
from torch import nn
from PIL import Image

from ram.models import ram_plus
from ram import inference_ram_openset as inference
from ram import get_transform
from ram.utils import build_openset_label_embedding

my_categories = [
    "cardboard box", "plastic tote", "bagged contents", "luggage",
    "large appliance", "small appliance", "electronics", "sofa",
    "chair", "table", "cabinet", "mattress", "tool chest",
    "bicycle", "fitness equipment", "clothing textile", "wall art"
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
transform = get_transform(image_size=384)

model = ram_plus(pretrained="pretrained/ram_plus_swin_large_14m.pth",
                  image_size=384, vit="swin_l")

openset_label_embedding, openset_categories = build_openset_label_embedding(my_categories)

model.tag_list = np.array(openset_categories)
model.label_embed = nn.Parameter(openset_label_embedding.float())
model.num_class = len(openset_categories)
model.class_threshold = torch.ones(model.num_class) * 0.5

model.eval()
model = model.to(device)

import sys
image_path = sys.argv[1]
image = transform(Image.open(image_path)).unsqueeze(0).to(device)

res = inference(image, model)
print("Image Tags:", res)

**Test:** Aşağıdaki dosya yolunu kendi dataset'inizdeki bir fotoğrafla değiştirin.

In [ ]:
IMG_PATH = "/kaggle/input/YOUR-DATASET/YOUR-IMAGE.jpg"  # <-- kendi yolunuzla değiştirin
!python inference_custom.py "{IMG_PATH}"

## Bölüm 2 — Grounded-SAM-2 Kurulumu (konum + maske)

RAM++'ın bulduğu etiketlerin **nerede** olduğunu (Grounding DINO) ve **piksel sınırını** (SAM2) çıkarıyoruz.

In [ ]:
%cd /kaggle/working
!git clone https://github.com/IDEA-Research/Grounded-SAM-2.git
%cd Grounded-SAM-2

SAM2'nin CUDA eklentisini derlemeden kuruyoruz (Kaggle'da bilinen derleme sorunlarını önlemek için); bu sadece küçük bir maske temizleme adımını devre dışı bırakır, ana sonucu etkilemez.

In [ ]:
import os
os.environ["SAM2_BUILD_CUDA"] = "0"
!pip install -e . -q

Grounding DINO'nun CUDA uzantısını `pip install -e` yerine doğrudan `setup.py build_ext` ile derliyoruz — Kaggle'da `pip install -e` bu paket için paketleme adımında başarısız oluyor, ama derlemenin kendisi doğrudan çalıştırılınca sorunsuz tamamlanıyor.

In [ ]:
os.environ["CUDA_HOME"] = ""
%cd /kaggle/working/Grounded-SAM-2/grounding_dino
!python setup.py build_ext --inplace
%cd /kaggle/working/Grounded-SAM-2

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/Grounded-SAM-2/grounding_dino")
import groundingdino
print("groundingdino başarıyla içe aktarıldı")

In [ ]:
%cd checkpoints
!bash download_ckpts.sh
%cd ../gdino_checkpoints
!bash download_ckpts.sh
%cd ..

Eksik bağımlılıkları tek seferde tamamlıyoruz.

In [ ]:
!pip install supervision addict yapf -q
!pip install -r /kaggle/working/Grounded-SAM-2/grounding_dino/requirements.txt -q
!pip install "transformers==4.46.3" -q

Kendi metin komutumuzla (sınıf listesi) çalışan script'i yazıyoruz. `TEXT_PROMPT` ve `IMG_PATH` değerlerini kendi ihtiyacınıza göre güncelleyin.

In [ ]:
%%writefile /kaggle/working/Grounded-SAM-2/grounded_sam2_bidvision.py
import os
import cv2
import json
import torch
import numpy as np
import supervision as sv
import pycocotools.mask as mask_util
from pathlib import Path
from torchvision.ops import box_convert
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from grounding_dino.groundingdino.util.inference import load_model, load_image, predict

TEXT_PROMPT = "cardboard box . plastic tote . bagged contents . luggage . large appliance . small appliance . electronics . sofa . chair . table . cabinet . mattress . tool chest . bicycle . fitness equipment . clothing textile . wall art ."
IMG_PATH = "/kaggle/input/YOUR-DATASET/YOUR-IMAGE.jpg"  # <-- kendi yolunuzla değiştirin
SAM2_CHECKPOINT = "./checkpoints/sam2.1_hiera_large.pt"
SAM2_MODEL_CONFIG = "configs/sam2.1/sam2.1_hiera_l.yaml"
GROUNDING_DINO_CONFIG = "grounding_dino/groundingdino/config/GroundingDINO_SwinT_OGC.py"
GROUNDING_DINO_CHECKPOINT = "gdino_checkpoints/groundingdino_swint_ogc.pth"
BOX_THRESHOLD = 0.35
TEXT_THRESHOLD = 0.25
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = Path("outputs/bidvision_test")
DUMP_JSON_RESULTS = True
MULTIMASK_OUTPUT = False

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sam2_model = build_sam2(SAM2_MODEL_CONFIG, SAM2_CHECKPOINT, device=DEVICE)
sam2_predictor = SAM2ImagePredictor(sam2_model)

grounding_model = load_model(
    model_config_path=GROUNDING_DINO_CONFIG,
    model_checkpoint_path=GROUNDING_DINO_CHECKPOINT,
    device=DEVICE
)

image_source, image = load_image(IMG_PATH)
sam2_predictor.set_image(image_source)

boxes, confidences, labels = predict(
    model=grounding_model, image=image, caption=TEXT_PROMPT,
    box_threshold=BOX_THRESHOLD, text_threshold=TEXT_THRESHOLD, device=DEVICE
)

h, w, _ = image_source.shape
boxes = boxes * torch.Tensor([w, h, w, h])
input_boxes = box_convert(boxes=boxes, in_fmt="cxcywh", out_fmt="xyxy").numpy()

torch.autocast(device_type=DEVICE, dtype=torch.bfloat16).__enter__()
if torch.cuda.is_available() and torch.cuda.get_device_properties(0).major >= 8:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

masks, scores, logits = sam2_predictor.predict(
    point_coords=None, point_labels=None, box=input_boxes, multimask_output=MULTIMASK_OUTPUT,
)
if masks.ndim == 4:
    masks = masks.squeeze(1)

confidences = confidences.numpy().tolist()
class_names = labels
class_ids = np.array(list(range(len(class_names))))
labels = [f"{c} {conf:.2f}" for c, conf in zip(class_names, confidences)]

img = cv2.imread(IMG_PATH)
detections = sv.Detections(xyxy=input_boxes, mask=masks.astype(bool), class_id=class_ids)

box_annotator = sv.BoxAnnotator()
annotated_frame = box_annotator.annotate(scene=img.copy(), detections=detections)
label_annotator = sv.LabelAnnotator()
annotated_frame = label_annotator.annotate(scene=annotated_frame, detections=detections, labels=labels)
cv2.imwrite(str(OUTPUT_DIR / "boxes.jpg"), annotated_frame)

mask_annotator = sv.MaskAnnotator()
annotated_frame = mask_annotator.annotate(scene=annotated_frame, detections=detections)
cv2.imwrite(str(OUTPUT_DIR / "masks.jpg"), annotated_frame)

print("Bulunan nesneler:", class_names)
print("Kaydedildi:", OUTPUT_DIR)

In [ ]:
!python grounded_sam2_bidvision.py

Sonuçları görüntüle:

In [ ]:
from IPython.display import Image as IPImage, display
display(IPImage("outputs/bidvision_test/boxes.jpg", width=700))

In [4]:
display(IPImage("outputs/bidvision_test/masks.jpg", width=700))

NameError: name 'IPImage' is not defined